# Análisis de señales previas al incumplimiento

Este notebook carga el archivo `ConsolidadoMonclova.xlsx`, genera métricas mensuales y diarias, y construye gráficos para detectar señales previas al incumplimiento de febrero de 2026.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, PercentFormatter
from pathlib import Path

In [ ]:
df = pd.read_excel('ConsolidadoMonclova.xlsx')
for col in ['fecha_operacion','fecha_vencimiento']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df['month'] = df['fecha_operacion'].dt.to_period('M').dt.to_timestamp()
df['volumen_usd_round'] = df['volumen_usd'].round(2)
df.head()

## Métricas mensuales

In [ ]:
monthly = df.groupby('month').agg(
    operaciones=('nm_operacion','count'),
    volumen_usd=('volumen_usd','sum'),
    ticket_usd=('volumen_usd','mean'),
    dias_activos=('fecha_operacion', lambda s: s.dt.normalize().nunique())
).reset_index()
for cls in ['SPOT','Prepago','Tiron']:
    temp = df[df['clasificacion']==cls].groupby('month').agg(**{f'usd_{cls.lower()}':('volumen_usd','sum')}).reset_index()
    monthly = monthly.merge(temp, on='month', how='left')
monthly = monthly.fillna(0)
monthly['share_usd_prepago'] = monthly['usd_prepago']/monthly['volumen_usd']
monthly['ops_por_dia'] = monthly['operaciones']/monthly['dias_activos']
monthly

## Concentración de tickets repetitivos

In [ ]:
ticket_patterns = []
for month, g in df.groupby('month'):
    ticket_patterns.append({
        'month': month,
        'share_usd_top2_tickets': g.loc[g['volumen_usd_round'].isin([1000000,1250000]), 'volumen_usd'].sum()/g['volumen_usd'].sum(),
        'share_ops_top2_tickets': g['volumen_usd_round'].isin([1000000,1250000]).mean(),
    })
ticket_patterns = pd.DataFrame(ticket_patterns).sort_values('month')
ticket_patterns

## Gráficos

In [ ]:
usd_fmt = FuncFormatter(lambda x, p: f'${x/1e6:,.1f}M')
fig, ax1 = plt.subplots(figsize=(11,5.5))
ax1.bar(monthly['month'], monthly['volumen_usd'], width=23)
ax1.set_title('Histórico mensual de volumen USD y número de operaciones')
ax1.set_ylabel('Volumen USD')
ax1.yaxis.set_major_formatter(usd_fmt)
ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['operaciones'], marker='o')
ax2.set_ylabel('Operaciones')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11,5.5))
ax.plot(monthly['month'], monthly['share_usd_prepago'], marker='o', label='% USD Prepago')
ax.plot(ticket_patterns['month'], ticket_patterns['share_usd_top2_tickets'], marker='o', label='% USD tickets 1M/1.25M')
ax.plot(monthly['month'], monthly['operaciones']/monthly['operaciones'].max(), marker='o', label='Operaciones (índice)')
ax.set_title('Señales de presión operativa / liquidez')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.show()

## Lectura sugerida

- Buscar máximos históricos en volumen, operaciones y días activos.
- Revisar el peso de Prepago/LDD como proxy de dependencia de fondeo.
- Medir la concentración en tickets repetitivos como señal de rollover o administración táctica de caja.
- Cruza este notebook con la disposición real del FX Loan para cerrar causalidad.